In [1]:
import os
import sys
from dotenv import load_dotenv
from utils.helper_functions import *
from utils.evaluate_rag import *

load_dotenv()

path="data/hyde_rag.pdf"

Custom Implementation

In [2]:
class HyDERetriever:
    def __init__(self,file_path,chunk_size=500,chunk_overlap=100):
        self.llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.6,max_completion_tokens=5000)
        self.embeddings=OpenAIEmbeddings(model='text-embedding-3-small')
        self.chunk_size=chunk_size
        self.chunk_overlap=chunk_overlap
        self.vectorstore=encode_pdf(file_path,self.chunk_size,self.chunk_overlap)

        self.HyDEPrompts=PromptTemplate(
            input_variables=["query","chunk_size"],
            template="""
                        You are an expert writer.
                        Given the following question, write a concise, factual document that would likely answer it.
                        The document should resemble a passage from a textbook, article, or knowledge base.
                        Do not mention that this is a hypothetical document.
                        Do not include phrases like "I think" or "As an AI".
                        the document size has be exactly {chunk_size} characters.

                    Question:
                    {query}
                    Hypothetical Document:
                    """
                    )
        self.HyDEChain=self.HyDEPrompts|self.llm

    def generate_hypothetical_documents(self,query:str):
        input_variable={"query":query,"chunk_size":self.chunk_size}
        return self.HyDEChain.invoke(input_variable).content

    def retrieve(self,query:str,k=3):
        hypothetical_docs=self.generate_hypothetical_documents(query)
        similar_docs=self.vectorstore.similarity_search(hypothetical_docs,k)
        return similar_docs,hypothetical_docs

        

In [4]:
retriever=HyDERetriever(path)
query="What happens when HyDE uses smaller instruction models like FLAN-T5 or Cohere?"
results,hypothetical_docs=retriever.retrieve(query)

print(f"{results}\n")
print(hypothetical_docs)

[Document(id='ab80a481-d238-4415-bcd1-34320e1a2116', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2022-12-21T01:43:04+00:00', 'author': '', 'keywords': '', 'moddate': '2022-12-21T01:43:04+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/hyde_rag.pdf', 'total_pages': 11, 'page': 5, 'page_label': '6'}, page_content='on TREC DL19/20.\n5.1 Effect of Different Generative Models\nIn Table 4, we show HyDE using other\ninstruction-following language models. In\nparticular, we consider a 52-billion Cohere\nmodel ( command-xlarge-20221108) and a\n11-billion FLAN model ( FLAN-T5-xxl; Wei\net al. (2022)). 2 Generally, we observe that all\n2Model sizes are from https://crfm.stanford.edu/\nhelm/v1.0/?models.\nModel DL19 DL20\nContriever 44.5 42.1\nContrieverFT 62.1 63.2\nHyDE\nw/ Contriever\nw/ Flan-T5 (11b) 48.9 52.9'), Doc

In [5]:
docs_content=[doc.page_content for doc in results]

show_context(docs_content)

Context:1
on TREC DL19/20.
5.1 Effect of Different Generative Models
In Table 4, we show HyDE using other
instruction-following language models. In
particular, we consider a 52-billion Cohere
model ( command-xlarge-20221108) and a
11-billion FLAN model ( FLAN-T5-xxl; Wei
et al. (2022)). 2 Generally, we observe that all
2Model sizes are from https://crfm.stanford.edu/
helm/v1.0/?models.
Model DL19 DL20
Contriever 44.5 42.1
ContrieverFT 62.1 63.2
HyDE
w/ Contriever
w/ Flan-T5 (11b) 48.9 52.9


Context:2
models bring improvement to the unsupervised
Contriever, with larger models bringing larger
improvements. At the time when this paper is
written, the Cohere model is still experimental
without much detail disclosed. We can only
tentatively hypothesize that training techniques
may have also played some role in the performance
difference.
5.2 HyDE with Fine-tuned Encoder
To begin with, HyDE with ﬁne-tuned encoder is
not the intended usage: HyDE is more powerful


Context:3
evance modeling a

Langchain Implementation

In [2]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import HypotheticalDocumentEmbedder

doc=PyPDFLoader(file_path="data/hyde_rag.pdf")

loaded_docs=doc.load()
chunker=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks=chunker.split_documents(loaded_docs)

prompt_template="web_search"             # web_search is a pre-built prompt template provided by langchain community

embedding=OpenAIEmbeddings()
llm=ChatOpenAI(model='gpt-4o-mini',temperature=0,max_completion_tokens=5000)
hyde_embeddings=HypotheticalDocumentEmbedder.from_llm(llm,embedding,prompt_template)

vectorstore=Chroma.from_documents(chunks,hyde_embeddings)
retriever=vectorstore.as_retriever( search_type="similarity", search_kwargs={"k": 3})

docs=retriever.invoke("What happens when HyDE uses smaller instruction models like FLAN-T5 or Cohere?")
doc_content=[doc.page_content for doc in docs]
show_context(doc_content)

Context:1
In Table 4, we show HyDE using other
instruction-following language models. In
particular, we consider a 52-billion Cohere
model ( command-xlarge-20221108) and a
11-billion FLAN model ( FLAN-T5-xxl; Wei
et al. (2022)). 2 Generally, we observe that all
2Model sizes are from https://crfm.stanford.edu/
helm/v1.0/?models.
Model DL19 DL20
Contriever 44.5 42.1
ContrieverFT 62.1 63.2
HyDE
w/ Contriever
w/ Flan-T5 (11b) 48.9 52.9
w/ Cohere (52b) 53.8 53.8
w/ GPT (175b) 61.3 57.9
w/ ContrieverFT
w/ Flan-T5 (11b) 60.2 62.1
w/ Cohere (52b) 61.4 63.1
w/ GPT (175b) 67.4 63.5
Table 4: NDCG@10 on TREC DL19/20. Effect
of changing different instruction LMs and using ﬁne-
tuned encoder. Best w/o relevance and overall models
are marked bold.
models bring improvement to the unsupervised
Contriever, with larger models bringing larger
improvements. At the time when this paper is
written, the Cohere model is still experimental
without much detail disclosed. We can only


Context:2
Contriever, with 

In [6]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser



llm=ChatOpenAI(temperature=0,model='gpt-4o-mini')
loader=PyPDFLoader(file_path="data/hyde_rag.pdf")
embedding=OpenAIEmbeddings(model='text-embedding-3-small')
chunker=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=500,separators=["\n\n", "\n", " ", ""])


print(f"Loading and Processing PDF ...")

loaded_pdfs=loader.load()
chunks=chunker.split_documents(loaded_pdfs)
print(f"Created {len(chunks)} Chunks from total {len(loaded_pdfs)}")

vectorstore=Chroma.from_documents(chunks,embedding)
retriever=vectorstore.as_retriever(search_kwargs={"k": 6})

# Helper Functions
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

def retrieve_docs(query):
    retireved_docs=retriever.invoke(query)
    return format_docs(retireved_docs)

def retrieve_docs_with_hyde(inputs):
    hypothetical_document=inputs["hypothetical"]
    print(f"Hypothetical Document :{hypothetical_document[:200]} \n")
    retrieve_docs=retriever.invoke(hypothetical_document)

    return {
        "context":retrieve_docs,
        "query":inputs["query"]
    }

qa_prompt = ChatPromptTemplate.from_template(
    """Use the following context to answer the question.

Context: {context}

Question: {question}

Answer:"""
)

# ============================================================================
# METHOD 1: STANDARD RAG (Direct Retrieval)
# ============================================================================

rag_chain=({"context":RunnableLambda(retrieve_docs),"query":RunnablePassthrough()}|qa_prompt|llm|StrOutputParser())
print(f"Standard Rag Chain Ready")


# ============================================================================
# METHOD 2: HYDE RAG (Hypothetical Document Retrieval)
# ============================================================================
print("\n🔧 Building HYDE RAG chain...")
hyde_prompt = ChatPromptTemplate.from_template(
    """Write a detailed, informative passage that would answer this question. 
Include relevant facts, explanations, and context.

Question: {question}

Passage:"""
)

hyde_chain=({"query":RunnablePassthrough(),"hypothetical":(hyde_prompt|llm|StrOutputParser())}|RunnableLambda(retrieve_docs_with_hyde)|qa_prompt|llm|StrOutputParser())
print(f"Hyde Rag Chain Ready")





Loading and Processing PDF ...
Created 77 Chunks from total 11
Standard Rag Chain Ready

🔧 Building HYDE RAG chain...
Hyde Rag Chain Ready
